# Phase 6 — Breaking the 95% Barrier: Alive-Cell Recall Attack

**Bottleneck:** Current best model (Deep ResConv 80f + PosEnc) reaches **90.6% pixel accuracy**, but alive-cell recall is only **72.8%** because the board is sparse (~27% alive) and BCE optimises the easy majority class.

## Two Novel Architectures

| | GoL Transformer (GoLT) | Physics-Informed CAN (PI-CAN) |
|---|---|---|
| **Key idea** | Self-attention over 100 cell tokens | Soft GoL physics loss in `train_step` |
| **Inductive bias** | Toroidal sinusoidal positional encoding | CBAM channel+spatial attention + torus padding |
| **Loss** | Combo: Tversky(α=0.3,β=0.7) + Focal(γ=2.5,α=0.65) | Same Combo + λ·BCE(softGoL(pred\_T-1), input\_T) |
| **Params** | ~300K | ~1.5M |

## Three New Weapons Against Class Imbalance

1. **Tversky Loss** (β=0.7 > α=0.3) — penalises FN (missed alive cells) **2.3× more** than FP; directly attacks recall
2. **Differentiable Physics Loss** — teaches the network: `soft_GoL(predicted T-1) ≈ observed T`; gradients flow through the approximated GoL rules
3. **Test-Time Augmentation (TTA)** — D4 ensemble (8 symmetry transforms) at inference; free ~0.5% gain

## Notebook Structure

| Cell | Content |
|---|---|
| 1–3 | Setup, data loading, EDA |
| 4–6 | Loss functions, metrics, differentiable GoL verification |
| 7–8 | Plot utilities |
| 9–12 | Architecture implementations |
| 13 | Training infrastructure (D4 augmentation, TTA, threshold search) |
| 14–19 | Phase 1 screening → Phase 2 full training → Phase 3 post-processing |
| 20–21 | Final comparison, error maps, per-pixel heatmap |

In [ ]:
import os, gc, time, math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score,
                             f1_score, confusion_matrix)

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)
tf.random.set_seed(42)

gpu = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | GPU: {gpu}')

In [ ]:
SIZE   = 10
GEN    = 2
AMOUNT = 100_000
PATH_DF = 'C:/GameOfLifeFiles/df/'
os.makedirs('models', exist_ok=True)

def load_data():
    pkl = f'{PATH_DF}{SIZE}-{AMOUNT}/{SIZE}size_{AMOUNT}boards_{GEN}gen_reverse.pkl'
    print(f'Loading {pkl} ...')
    df  = pd.read_pickle(pkl)
    nf  = (GEN - 1) * SIZE * SIZE
    X   = df.iloc[:, :nf].to_numpy(np.float32).reshape(-1, SIZE, SIZE, 1)
    y   = df.iloc[:, nf:nf + SIZE*SIZE].to_numpy(np.float32).reshape(-1, SIZE, SIZE, 1)
    del df; gc.collect()
    i_tv, i_te = train_test_split(np.arange(len(X)), test_size=.1, random_state=365)
    i_tr, i_va = train_test_split(i_tv,              test_size=.1, random_state=365)
    d = dict(X_train=X[i_tr], y_train=y[i_tr],
             X_val  =X[i_va], y_val  =y[i_va],
             X_test =X[i_te], y_test =y[i_te])
    ar = float(y[i_tr].mean())
    print(f'  train={len(i_tr):,}  val={len(i_va):,}  test={len(i_te):,}')
    print(f'  alive_ratio={ar:.3f}   dead_ratio={1-ar:.3f}')
    return d, ar

data, ALIVE_RATIO = load_data()
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

# 1. Alive-cell count distribution per board
alive_cnt = y_train.reshape(-1, SIZE*SIZE).sum(axis=1)
axes[0].hist(alive_cnt, bins=25, color='steelblue', edgecolor='white', alpha=.85)
axes[0].axvline(alive_cnt.mean(), c='red', ls='--', lw=2,
                label=f'mean = {alive_cnt.mean():.1f}')
axes[0].set(xlabel='Alive cells per T-1 board', ylabel='Boards',
            title='Alive Cell Count Distribution (training set)')
axes[0].legend()

# 2. Pixel class balance
alive_n = int(y_train.sum())
dead_n  = int((1 - y_train).sum())
axes[1].pie([dead_n, alive_n], labels=['Dead', 'Alive'],
            colors=['#5B8DB8', '#E85D04'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Pixel Class Balance (training set)')

# 3. Alive neighbour count (proxy for GoL difficulty)
neigh_alive = []
for b in y_train[:5000]:
    board  = b[:, :, 0]
    padded = np.pad(board, 1, mode='wrap')
    for i in range(SIZE):
        for j in range(SIZE):
            if board[i, j] == 1:
                neigh_alive.append(padded[i:i+3, j:j+3].sum() - 1)
axes[2].hist(neigh_alive, bins=range(10), color='#FF5722', edgecolor='white', alpha=.85)
axes[2].set(xlabel='Alive neighbours of an alive cell', ylabel='Count',
            title='Neighbourhood Context\n(T-1 alive cells, first 5K boards)')

plt.tight_layout(); plt.show()

# Sample board pairs
n_s  = 6
idxs = np.random.RandomState(0).choice(len(X_train), n_s, replace=False)
fig2, ax2 = plt.subplots(2, n_s, figsize=(n_s * 2.3, 5))
fig2.suptitle('Sample Pairs — Input T (top)  vs  Target T-1 (bottom)', fontsize=11)
for col, si in enumerate(idxs):
    ax2[0, col].imshow(X_train[si, :, :, 0], cmap='binary', vmin=0, vmax=1,
                       interpolation='nearest')
    ax2[0, col].axis('off')
    ax2[0, col].set_title(f'T  [#{si}]', fontsize=8)
    ax2[1, col].imshow(y_train[si, :, :, 0], cmap='binary', vmin=0, vmax=1,
                       interpolation='nearest')
    ax2[1, col].axis('off')
    ax2[1, col].set_title(f'T-1 [#{si}]', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ════════════════════════════════════════════════════════
# Loss Functions
# ════════════════════════════════════════════════════════

def focal_loss(gamma=2.0, alpha=0.6):
    '''Focal loss: alpha weights alive class, gamma down-weights easy pixels.'''
    def _loss(yt, yp):
        yp = tf.clip_by_value(yp, 1e-7, 1. - 1e-7)
        at = yt * alpha + (1. - yt) * (1. - alpha)
        pt = yt * yp + (1. - yt) * (1. - yp)
        bce = -(yt * tf.math.log(yp) + (1. - yt) * tf.math.log(1. - yp))
        return tf.reduce_mean(at * tf.pow(1. - pt, gamma) * bce)
    return _loss

def tversky_loss(alpha=0.3, beta=0.7):
    '''Tversky Loss (generalised Dice). beta > alpha => FN penalised more than FP.'''
    def _loss(yt, yp):
        yf = tf.reshape(yt, [-1]); pf = tf.reshape(yp, [-1])
        TP = tf.reduce_sum(yf * pf)
        FP = tf.reduce_sum((1. - yf) * pf)
        FN = tf.reduce_sum(yf * (1. - pf))
        return 1. - (TP + 1e-6) / (TP + alpha*FP + beta*FN + 1e-6)
    return _loss

def combo_loss(tv_a=0.3, tv_b=0.7, f_g=2.5, f_a=0.65, r=0.5):
    '''Combo = r*Tversky + (1-r)*Focal. Tversky drives recall, Focal prevents collapse.'''
    tv = tversky_loss(tv_a, tv_b); fl = focal_loss(f_g, f_a)
    def _loss(yt, yp): return r * tv(yt, yp) + (1. - r) * fl(yt, yp)
    return _loss

LOSS_FN = combo_loss()

# ── Visualise loss landscape for an alive cell (y_true=1) ───────────────────
pv  = np.linspace(0.01, 0.99, 200).reshape(-1, 1, 1, 1).astype(np.float32)
ones = np.ones_like(pv)
loss_curves = {
    'BCE (baseline)':      [float(tf.reduce_mean(
                               tf.keras.losses.binary_crossentropy(ones, p))) for p in pv],
    'Focal(γ=2,α=0.6)':   [focal_loss(2.0, 0.6)(ones, p).numpy()  for p in pv],
    'Tversky(α=.3,β=.7)': [tversky_loss(.3, .7)(ones, p).numpy()  for p in pv],
    'Combo (default)':     [combo_loss()(ones, p).numpy()           for p in pv],
}
fig, ax = plt.subplots(figsize=(8, 4))
styles = [('gray', '--'), ('#2196F3', '-'), ('#FF5722', '-'), ('#4CAF50', '-')]
for (name, vals), (c, ls) in zip(loss_curves.items(), styles):
    ax.plot(pv[:, 0, 0, 0], vals, lw=2, c=c, ls=ls, label=name)
ax.axvline(.5, c='gray', ls=':', alpha=.4)
ax.set(xlabel='Predicted probability (for alive cell, y_true=1)',
       ylabel='Loss value',
       title='Loss Landscape for Alive Cells — Tversky/Combo penalise low confidence more',
       xlim=[0, 1])
ax.legend(fontsize=9); ax.grid(True, alpha=.3)
plt.tight_layout(); plt.show()

# ════════════════════════════════════════════════════════
# Custom Metrics — alive-class precision / recall / F1
# ════════════════════════════════════════════════════════

class _CM(tf.keras.metrics.Metric):
    def __init__(self, t=0.5, name='m', **kw):
        super().__init__(name=name, **kw)
        self.t  = t
        self.tp = self.add_weight('tp', initializer='zeros')
        self.pp = self.add_weight('pp', initializer='zeros')
        self.ap = self.add_weight('ap', initializer='zeros')
    def update_state(self, yt, yp, sw=None):
        yp = tf.cast(yp > self.t, tf.float32)
        yt = tf.cast(yt > .5,     tf.float32)
        self.tp.assign_add(tf.reduce_sum(yt * yp))
        self.pp.assign_add(tf.reduce_sum(yp))
        self.ap.assign_add(tf.reduce_sum(yt))
    def reset_state(self): [v.assign(0.) for v in (self.tp, self.pp, self.ap)]

class AlivePrecision(_CM):
    def __init__(self, **kw): super().__init__(name='alive_prec', **kw)
    def result(self): return self.tp / (self.pp + 1e-7)

class AliveRecall(_CM):
    def __init__(self, **kw): super().__init__(name='alive_rec', **kw)
    def result(self): return self.tp / (self.ap + 1e-7)

class AliveF1(_CM):
    def __init__(self, **kw): super().__init__(name='alive_f1', **kw)
    def result(self):
        p = self.tp / (self.pp + 1e-7); r = self.tp / (self.ap + 1e-7)
        return 2. * p * r / (p + r + 1e-7)

def alive_metrics():
    return [tf.keras.metrics.BinaryAccuracy(name='acc'),
            AlivePrecision(), AliveRecall(), AliveF1()]

print('Loss functions and custom metrics ready.')

In [ ]:
def soft_gol_step(p, T=10.0):
    '''
    Differentiable approximation of Conway's Game of Life forward step.

    Sigmoid gates replace the hard if/else GoL rules so that gradients
    can flow back through the GoL computation into the backbone weights.

    Survival gate:  alive & n in {2,3}  =>  sigmoid(T*(n-1.5)) * sigmoid(T*(3.5-n))
    Birth gate:     dead  & n == 3       =>  sigmoid(T*(n-2.5)) * sigmoid(T*(3.5-n))

    p : (B, 10, 10, 1) float32 probabilities in [0, 1]
    Returns next-state probabilities, same shape.
    '''
    # Toroidal padding: 1px on each side -> (B, 12, 12, 1)
    pp = tf.concat([p[:, -1:, :, :], p, p[:, :1, :, :]], axis=1)
    pp = tf.concat([pp[:, :, -1:, :], pp, pp[:, :, :1, :]], axis=2)
    # All-ones 3x3 kernel -> sum 9 cells, subtract centre -> neighbour count
    k  = tf.ones([3, 3, 1, 1], p.dtype)
    n  = tf.nn.conv2d(pp, k, strides=[1, 1, 1, 1], padding='VALID') - p
    survive = p       * tf.sigmoid(T*(n - 1.5)) * tf.sigmoid(T*(3.5 - n))
    birth   = (1. - p) * tf.sigmoid(T*(n - 2.5)) * tf.sigmoid(T*(3.5 - n))
    return survive + birth

# ── Verify: soft GoL vs reference hard GoL ──────────────────────────────────
def _hard_gol_np(b):
    '''Reference numpy GoL forward step (toroidal).'''
    b = b.astype(np.float32)
    pad = np.pad(b, ((0,0),(1,1),(1,1)), mode='wrap')
    n = sum(pad[:, 1+di:11+di, 1+dj:11+dj]
            for di in (-1,0,1) for dj in (-1,0,1)
            if not (di==0 and dj==0))
    alive = b.astype(bool)
    return ((alive & ((n==2)|(n==3))) | (~alive & (n==3))).astype(np.float32)

X3     = X_test[:, :, :, 0]
hard   = _hard_gol_np(X3)
soft   = soft_gol_step(tf.constant(X_test)).numpy()[..., 0]
soft_b = (soft > .5).astype(np.float32)
agree  = float(np.mean(soft_b == hard))
print(f'Soft GoL (T=10) matches hard GoL on {agree:.4%} of cells')

# Visualise one sample
idx_v  = 7
titles = ['Input (X_t)', 'Hard GoL step', 'Soft GoL (prob)', 'Soft GoL (binary)', 'Agreement']
cmaps  = ['binary',       'binary',        'RdYlGn',           'binary',            'RdYlGn']
imgs   = [X_test[idx_v,:,:,0], hard[idx_v], soft[idx_v], soft_b[idx_v],
          (soft_b[idx_v] == hard[idx_v]).astype(np.float32)]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle(f'Differentiable GoL Verification  (cell agreement = {agree:.4%})',
             fontsize=11, fontweight='bold')
for ax, img, t, cm in zip(axes, imgs, titles, cmaps):
    im = ax.imshow(img, cmap=cm, vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(t, fontsize=9); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=.05, pad=.03)
plt.tight_layout(); plt.show()

In [ ]:
def plot_history(histories, names, figsize=(14, 9)):
    '''4-panel training history. Dashed = train, solid = val.'''
    cmap   = plt.cm.tab10(np.linspace(0, .9, max(len(histories), 2)))
    panels = [('loss',       'val_loss',       'Loss (lower = better)'),
              ('acc',        'val_acc',         'Pixel Accuracy'),
              ('alive_rec',  'val_alive_rec',   'Alive Recall  ← the bottleneck'),
              ('alive_f1',   'val_alive_f1',    'Alive F1')]
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Training History', fontsize=13, fontweight='bold')
    for ax, (tr_k, va_k, title) in zip(axes.flat, panels):
        for h, name, c in zip(histories, names, cmap):
            ep = range(1, len(h.history[tr_k]) + 1)
            ax.plot(ep, h.history[tr_k], '--', color=c, alpha=.35, lw=1)
            ax.plot(ep, h.history[va_k], '-',  color=c, label=name, lw=2)
        ax.set_title(title); ax.set_xlabel('Epoch')
        ax.legend(fontsize=8); ax.grid(True, alpha=.3)
    plt.tight_layout(); plt.show()


def plot_threshold_curves(probs_dict, y_true, title='Threshold Search'):
    '''Precision / Recall / F1 vs threshold for multiple models side by side.'''
    thresholds = np.arange(.20, .81, .01)
    yt = y_true.flatten().astype(int)
    n  = len(probs_dict)
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4), squeeze=False)
    fig.suptitle(title, fontsize=12, fontweight='bold')
    best_ts = {}
    for ax, (name, prob) in zip(axes[0], probs_dict.items()):
        pf = prob.flatten()
        precs = [precision_score(yt, (pf>t).astype(int), zero_division=0) for t in thresholds]
        recs  = [recall_score(   yt, (pf>t).astype(int), zero_division=0) for t in thresholds]
        f1s   = [f1_score(       yt, (pf>t).astype(int), zero_division=0) for t in thresholds]
        bi    = int(np.argmax(f1s))
        ax.plot(thresholds, precs, 'b-', lw=2, label='Precision')
        ax.plot(thresholds, recs,  'r-', lw=2, label='Recall')
        ax.plot(thresholds, f1s,   'g-', lw=2, label='F1', zorder=5)
        ax.axvline(thresholds[bi], c='green', ls='--', alpha=.7,
                   label=f'Best t={thresholds[bi]:.2f}  F1={f1s[bi]:.4f}')
        ax.set(xlabel='Threshold', ylabel='Score', title=name,
               xlim=[.2, .8], ylim=[0, 1])
        ax.legend(fontsize=8); ax.grid(True, alpha=.3)
        best_ts[name] = float(thresholds[bi])
    plt.tight_layout(); plt.show()
    return best_ts


def plot_comparison(results, title='Model Comparison', baseline=0.9059):
    '''Grouped bar chart across models; red dashed line = baseline.'''
    models  = list(results.keys())
    metrics = [('pixel_acc', 'Pixel Acc'), ('alive_rec', 'Alive Recall'),
               ('alive_prec','Alive Prec'), ('alive_f1', 'Alive F1')]
    cols    = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']
    x = np.arange(len(models)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(8, len(models)*2.8), 5))
    for i, ((mk, ml), c) in enumerate(zip(metrics, cols)):
        vals = [results[m].get(mk, 0) for m in models]
        bars = ax.bar(x + i*w, vals, w, label=ml, color=c, alpha=.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + .005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    if baseline:
        ax.axhline(baseline, c='red',     ls='--', alpha=.55, lw=1.5, label=f'Baseline {baseline:.3f}')
    ax.axhline(.95, c='darkred', ls=':',  alpha=.4, lw=1.5, label='95% target')
    ax.set(xticks=x + w*1.5, ylabel='Score', ylim=[0, 1.1], title=title)
    ax.set_xticklabels(models, rotation=20, ha='right', fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, axis='y', alpha=.3)
    plt.tight_layout(); plt.show()


def visualize_predictions(X, y_true, pred_dict, n=5, seed=0):
    '''
    Error-map grid.
    White=TP (correct alive)   Black=TN (correct dead)
    Orange=FP (false alive)    Blue=FN (missed alive — the main problem)
    '''
    emap = {(0,0): [.08,.08,.08], (1,1): [1.,1.,1.],
            (0,1): [1.,.55,.1],   (1,0): [.2,.5,1.]}
    rng  = np.random.RandomState(seed)
    idxs = rng.choice(len(X), n, replace=False)
    nc   = 2 + len(pred_dict)
    fig, axes = plt.subplots(n, nc, figsize=(nc*2.5, n*2.5))
    if n == 1: axes = axes[np.newaxis, :]
    for j, h in enumerate(['Input T', 'True T-1'] + list(pred_dict.keys())):
        axes[0, j].set_title(h, fontsize=9, fontweight='bold')
    for i, si in enumerate(idxs):
        axes[i, 0].imshow(X[si,:,:,0], cmap='binary', vmin=0, vmax=1,
                          interpolation='nearest'); axes[i, 0].axis('off')
        axes[i, 1].imshow(y_true[si,:,:,0], cmap='binary', vmin=0, vmax=1,
                          interpolation='nearest'); axes[i, 1].axis('off')
        for j, (mn, yp) in enumerate(pred_dict.items()):
            pb  = (yp[si,:,:,0] > .5).astype(int)
            tb  = (y_true[si,:,:,0] > .5).astype(int)
            rgb = np.array([[emap[(tb[r,c], pb[r,c])]
                             for c in range(SIZE)] for r in range(SIZE)])
            axes[i, j+2].imshow(rgb, interpolation='nearest'); axes[i, j+2].axis('off')
    patches = [mpatches.Patch(color=emap[(1,1)], label='TP (correct alive)', ec='black'),
               mpatches.Patch(color=emap[(0,0)], label='TN (correct dead)'),
               mpatches.Patch(color=emap[(0,1)], label='FP (false alive)'),
               mpatches.Patch(color=emap[(1,0)], label='FN (missed alive)')]
    fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=8,
               bbox_to_anchor=(.5, -.01))
    fig.suptitle('Prediction Error Maps', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, .04, 1, 1]); plt.show()


def evaluate_model(model, X, y, label='', threshold=.5, use_tta=False):
    '''Return metrics dict + raw probability array.'''
    if use_tta:
        yp = predict_tta(model, X)
    else:
        yp = model.predict(X, batch_size=512, verbose=0)
    yb  = (yp > threshold).astype(int)
    yi  = (y > .5).astype(int)
    fp  = yb.flatten(); ft = yi.flatten()
    am  = ft == 1; dm = ft == 0
    res = dict(pixel_acc  = float(np.mean(fp==ft)),
               dead_rec   = float(np.mean(fp[dm]==0)) if dm.any() else 0.,
               alive_rec  = float(np.mean(fp[am]==1)) if am.any() else 0.,
               alive_prec = precision_score(ft, fp, zero_division=0),
               alive_f1   = f1_score(ft, fp, zero_division=0),
               boards_90  = float(np.mean(
                   np.mean(yb.reshape(-1, SIZE*SIZE) == yi.reshape(-1, SIZE*SIZE),
                           axis=1) >= .9)),
               y_prob=yp)
    tta = ' TTA' if use_tta else ''
    print(f'  [{label}]{tta}  t={threshold:.2f}'
          f'  acc={res["pixel_acc"]:.4f}'
          f'  alive_rec={res["alive_rec"]:.4f}'
          f'  alive_prec={res["alive_prec"]:.4f}'
          f'  f1={res["alive_f1"]:.4f}'
          f'  boards>=90%={res["boards_90"]:.2%}')
    return res

print('Plot utilities ready.')

---

## Architecture 1 — GoL Transformer (GoLT)

```
Input (10×10×1)
  ↓ Toroidal pad (1px) + Conv(32, 3×3, valid) + BN + ReLU   ← CNN tokeniser
  ↓ Conv(64, 3×3, same) + BN + ReLU                          ← each cell = 64-dim feature vector
  ↓ Reshape → 100 tokens × 64 dims
  ↓ + Toroidal positional embedding (circular harmonics, learnable)
  ↓ 4 × Transformer blocks (pre-norm, 4 heads, GELU FFN)
  ↓ LayerNorm → Dense(1, sigmoid) per token
  ↓ Reshape → (10×10×1)
```

**Why a Transformer?**  
Current best models are CNNs — every cell only interacts with its neighbours *per layer*. Over 7 residual blocks, distant cells interact only indirectly. GoL patterns like gliders and oscillators create subtle long-range dependencies that full self-attention captures *in one layer*.

**CNN tokeniser (not raw cell value):**  
Instead of projecting a single bit to 64 dims, a small CNN first extracts the 3×3 neighbourhood context. The Transformer then reasons about *enriched cell representations*, not raw binary values. This retains the CNN's local-structure inductive bias.

**Toroidal positional encoding:**  
Standard 2-D sinusoids represent position (0,0) and (0,9) as far apart. On a 10×10 torus they are neighbours. Circular harmonics `sin(k · 2π · row/10)`, `cos(...)` correctly place them close in embedding space.

In [ ]:
def _torus_pos_enc(board_size=10, embed_dim=64):
    '''Circular 2-D positional encoding for a toroidal board.'''
    n    = board_size * board_size
    half = embed_dim // 2
    nf   = half // 2          # frequency bands per spatial dim
    enc  = np.zeros((n, embed_dim), dtype=np.float32)
    for idx in range(n):
        r, c = divmod(idx, board_size)
        for k in range(nf):
            freq = (k + 1) * 2. * math.pi / board_size
            enc[idx, 4*k    ] = math.sin(r * freq)
            enc[idx, 4*k + 1] = math.cos(r * freq)
            enc[idx, 4*k + 2] = math.sin(c * freq)
            enc[idx, 4*k + 3] = math.cos(c * freq)
    return enc   # (100, 64)


def _transformer_block(x, heads, dim, mlp_ratio=4, drop=.10):
    '''Pre-LN Transformer block (more stable than post-LN for small models).'''
    xn   = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    attn = tf.keras.layers.MultiHeadAttention(
               num_heads=heads, key_dim=dim//heads, dropout=drop)(xn, xn)
    x    = x + attn
    xn   = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    ffn  = tf.keras.layers.Dense(dim * mlp_ratio, activation='gelu')(xn)
    ffn  = tf.keras.layers.Dropout(drop)(ffn)
    ffn  = tf.keras.layers.Dense(dim)(ffn)
    ffn  = tf.keras.layers.Dropout(drop)(ffn)
    return x + ffn


def build_gol_transformer(board_size=10, embed_dim=64, heads=4,
                           n_layers=4, mlp_ratio=4, drop=.10, pad=1):
    inp = tf.keras.Input((board_size, board_size, 1), name='board_T')

    # CNN tokeniser with toroidal padding
    x = tf.keras.layers.Lambda(
            lambda t: tf.concat([t[:,-pad:,:,:], t, t[:,:pad,:,:]], axis=1))(inp)
    x = tf.keras.layers.Lambda(
            lambda t: tf.concat([t[:,:,-pad:,:], t, t[:,:,:pad,:]], axis=2))(x)
    x = tf.keras.layers.Conv2D(embed_dim//2, 3, padding='valid',
                                use_bias=False, name='tok_conv1')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(embed_dim, 3, padding='same',
                                use_bias=False, name='tok_conv2')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    # (B, 10, 10, embed_dim) → (B, 100, embed_dim)
    x = tf.keras.layers.Reshape((board_size*board_size, embed_dim))(x)

    # Toroidal positional embedding (learnable, initialised from circular harmonics)
    pos_init = _torus_pos_enc(board_size, embed_dim)
    pos_emb  = tf.keras.layers.Embedding(
                   board_size*board_size, embed_dim,
                   embeddings_initializer=tf.keras.initializers.Constant(pos_init),
                   name='pos_embed')
    x = x + pos_emb(tf.range(board_size*board_size))

    # Transformer encoder
    for _ in range(n_layers):
        x = _transformer_block(x, heads, embed_dim, mlp_ratio, drop)

    x   = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x   = tf.keras.layers.Dense(1, activation='sigmoid', name='cell_logit')(x)
    out = tf.keras.layers.Reshape((board_size, board_size, 1),
                                   name='output')(x)
    return tf.keras.Model(inp, out, name='GoLTransformer')


# Build and summarise
golt_proto = build_gol_transformer()
golt_proto.summary()
print(f'GoL Transformer params: {golt_proto.count_params():,}')

# Visualise toroidal positional encoding (first 4 dims)
pos_enc = _torus_pos_enc(10, 64).reshape(10, 10, 64)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle('Toroidal Positional Encoding — first 4 dimensions\n'
             '(circular pattern wraps correctly at edges)', fontsize=10)
for i, ax in enumerate(axes):
    im = ax.imshow(pos_enc[:, :, i], cmap='RdBu', interpolation='nearest')
    ax.set_title(f'dim {i}'); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=.05)
plt.tight_layout(); plt.show()
del golt_proto

---

## Architecture 2 — Physics-Informed CellAttentionNet (PI-CAN)

```
Input (10×10×1)
  ↓ Toroidal wrap padding (pad=2)       ← every 3×3 conv sees correct toroidal neighbours
  ↓ Entry conv (96 filters, 3×3)
  ↓ 8 × Residual CBAM blocks            ← channel attn + spatial attn inside each block
  ↓ Crop back to 10×10
  ↓ 1×1 conv → sigmoid
  [During training only:]
  ↓ soft_GoL(predicted_T-1) → BCE vs input_T   ← physics constraint
```

**CBAM = Convolutional Block Attention Module**  
Applied *inside* each residual block after the second conv:
- **Channel attention** (Squeeze-and-Excite): global average + max pooling → shared MLP → sigmoid weights per filter channel. Amplifies filters that encode alive-cell patterns.
- **Spatial attention**: per-position importance map from channel-reduced feature stats. Points the network at the sparse alive regions.

**Physics-Informed Training (custom `train_step`)**

```
L_total = L_combo(y_pred, y_true_T-1)
        + λ · BCE( soft_GoL(y_pred), x_T )
```

`soft_GoL` is fully differentiable, so gradients flow backward through the approximated GoL rules into the backbone weights. This teaches the model not just to match T-1 labels, but to produce predictions that are **physically consistent** with the observed T board — an additional free signal that existing models completely ignore.

In [ ]:
def _chan_attn(x, ratio=8):
    '''Squeeze-and-Excite channel attention (shared MLP on avg + max pools).'''
    C  = x.shape[-1]; r = max(1, C // ratio)
    d1 = tf.keras.layers.Dense(r, activation='relu',    use_bias=False)
    d2 = tf.keras.layers.Dense(C, activation='sigmoid', use_bias=False)
    ag = tf.keras.layers.GlobalAveragePooling2D(keepdims=True)(x)
    mx = tf.keras.layers.GlobalMaxPooling2D(keepdims=True)(x)
    return tf.keras.layers.Multiply()([x, d2(d1(ag)) + d2(d1(mx))])

def _spat_attn(x, ks=7):
    '''Spatial attention: importance map from channel-reduced stats.'''
    avg  = tf.reduce_mean(x, axis=-1, keepdims=True)
    mx   = tf.reduce_max(x,  axis=-1, keepdims=True)
    attn = tf.keras.layers.Conv2D(1, ks, padding='same', activation='sigmoid')(
               tf.keras.layers.Concatenate(axis=-1)([avg, mx]))
    return tf.keras.layers.Multiply()([x, attn])

def _res_cbam(x, f, drop=.10):
    '''Residual block with CBAM (channel + spatial attention).'''
    sc = (x if x.shape[-1] == f else
          tf.keras.layers.BatchNormalization()(
              tf.keras.layers.Conv2D(f, 1, use_bias=False)(x)))
    x  = tf.keras.layers.Conv2D(f, 3, padding='same', use_bias=False,
             kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x  = tf.keras.layers.BatchNormalization()(x)
    x  = tf.keras.layers.Activation('relu')(x)
    x  = tf.keras.layers.SpatialDropout2D(drop)(x)
    x  = tf.keras.layers.Conv2D(f, 3, padding='same', use_bias=False,
             kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x  = tf.keras.layers.BatchNormalization()(x)
    x  = _chan_attn(x)   # channel attention
    x  = _spat_attn(x)  # spatial attention
    return tf.keras.layers.Activation('relu')(
               tf.keras.layers.Add()([x, sc]))

def _wrap(t, p):
    '''Toroidal wrap padding.'''
    t = tf.concat([t[:,-p:,:,:], t, t[:,:p,:,:]], axis=1)
    return tf.concat([t[:,:,-p:,:], t, t[:,:,:p,:]], axis=2)

def build_can(board_size=10, filters=96, n_blocks=8, pad=2, drop=.10):
    inp = tf.keras.Input((board_size, board_size, 1), name='board_T')
    x   = tf.keras.layers.Lambda(lambda t: _wrap(t, pad), name='torus_pad')(inp)
    x   = tf.keras.layers.Conv2D(filters, 3, padding='same',
                                  use_bias=False, name='entry')(x)
    x   = tf.keras.layers.BatchNormalization()(x)
    x   = tf.keras.layers.Activation('relu')(x)
    for _ in range(n_blocks):
        x = _res_cbam(x, filters, drop)
    x   = tf.keras.layers.Cropping2D(((pad, pad), (pad, pad)), name='crop')(x)
    out = tf.keras.layers.Conv2D(1, 1, activation='sigmoid', name='output')(x)
    return tf.keras.Model(inp, out, name='CellAttentionNet')


class PhysicsInformedModel(tf.keras.Model):
    '''
    Wraps any backbone model with a physics-informed train_step.

    Extra loss during training:
        L_physics = BCE( soft_GoL(predicted_T-1), input_T )

    Because soft_GoL is differentiable, gradients flow back through the
    approximated GoL rules, teaching the backbone to produce GoL-valid boards.
    '''
    def __init__(self, backbone, phys_w=.15, **kw):
        super().__init__(**kw)
        self.backbone = backbone
        self.phys_w   = phys_w
        self._pl  = tf.keras.metrics.Mean(name='phys_loss')
        self._prl = tf.keras.metrics.Mean(name='prim_loss')

    def call(self, x, training=None):
        return self.backbone(x, training=training)

    @property
    def metrics(self):
        return super().metrics + [self._pl, self._prl]

    def train_step(self, data):
        x, y = data   # x = board T (input),  y = board T-1 (target)
        with tf.GradientTape() as tape:
            yp   = self(x, training=True)
            prim = self.compiled_loss(y, yp)
            # Physics: soft_GoL(predicted T-1) should reproduce observed T
            phys = tf.reduce_mean(
                       tf.keras.losses.binary_crossentropy(x, soft_gol_step(yp)))
            loss = prim + self.phys_w * phys
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.compiled_metrics.update_state(y, yp)
        self._pl.update_state(phys); self._prl.update_state(prim)
        r = {m.name: m.result() for m in self.metrics}
        r['loss'] = loss; return r

    def test_step(self, data):
        x, y = data
        yp = self(x, training=False)
        self.compiled_loss(y, yp)
        self.compiled_metrics.update_state(y, yp)
        return {m.name: m.result() for m in self.metrics}


# Build + summarise
can_proto = build_can()
can_proto.summary()
print(f'CellAttentionNet params: {can_proto.count_params():,}')
del can_proto

In [ ]:
class D4Seq(tf.keras.utils.Sequence):
    '''On-the-fly D4 dihedral augmentation (random of 8 GoL-symmetric transforms).'''
    def __init__(self, X, y, bs=512):
        self.X, self.y, self.bs = X, y, bs
        self._shuffle()
    def _shuffle(self):
        i = np.random.permutation(len(self.X))
        self.X, self.y = self.X[i], self.y[i]
    def __len__(self): return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, idx):
        s = idx * self.bs; e = min(s + self.bs, len(self.X))
        Xb = self.X[s:e].copy(); yb = self.y[s:e].copy()
        k = np.random.randint(4); fl = np.random.rand() > .5
        if k:  Xb = np.rot90(Xb, k, (1,2)); yb = np.rot90(yb, k, (1,2))
        if fl: Xb = Xb[:,:,::-1,:];          yb = yb[:,:,::-1,:]
        return Xb, yb
    def on_epoch_end(self): self._shuffle()


def predict_tta(model, X, bs=512):
    '''
    D4 Test-Time Augmentation: average predictions over all 8 transforms.
    Each prediction is inverse-transformed before averaging.
    GoL is symmetric under all 8 D4 transforms, so all views are equally valid.
    '''
    preds = []
    for k in range(4):
        for fl in (False, True):
            Xt = np.rot90(X, k=k, axes=(1,2))
            if fl: Xt = Xt[:,:,::-1,:]
            p = model.predict(Xt, batch_size=bs, verbose=0)
            if fl: p = p[:,:,::-1,:]
            p = np.rot90(p, k=-k, axes=(1,2))
            preds.append(p)
    return np.mean(preds, axis=0)


def find_best_threshold(y_prob, y_true, metric='f1'):
    '''Grid search [0.20, 0.75] for best F1 or recall threshold.'''
    fp = y_prob.flatten(); ft = y_true.flatten().astype(int)
    best_t, best_s = .5, 0.
    for t in np.arange(.20, .76, .01):
        pred = (fp > t).astype(int)
        s = (f1_score(ft, pred, zero_division=0) if metric == 'f1'
             else recall_score(ft, pred, zero_division=0))
        if s > best_s: best_s, best_t = s, t
    return float(best_t), float(best_s)


def run_phase(name, model_fn, X_tr, y_tr, X_va, y_va,
              loss_fn, lr=1e-3, epochs=30, bs=512, aug=True, phys_w=0.):
    '''
    Build → (optionally wrap) → compile → train → return (wrapped_model, best_f1, history).
    model_fn is a callable that returns a fresh model (ensures clean weights).
    '''
    tf.keras.backend.clear_session()
    model = model_fn()
    if phys_w > 0:
        wrapped = PhysicsInformedModel(model, phys_w=phys_w, name=f'{name}_pi')
    else:
        wrapped = model
    wrapped.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4),
        loss=loss_fn, metrics=alive_metrics())
    cbs = [
        tf.keras.callbacks.EarlyStopping('val_alive_f1', patience=12,
            restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=.5, patience=4,
            min_lr=5e-7, verbose=1),
    ]
    if aug:
        h = wrapped.fit(D4Seq(X_tr, y_tr, bs), validation_data=(X_va, y_va),
                        epochs=epochs, callbacks=cbs, verbose=2)
    else:
        h = wrapped.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                        epochs=epochs, batch_size=bs, callbacks=cbs, verbose=2)
    best_f1 = max(h.history.get('val_alive_f1', [0.]))
    return wrapped, best_f1, h

print('Training infrastructure ready.')